# 🧬 Comprehensive Paralog Synthetic Lethality Scanner
## DepMap Public 25Q3 - All Categories Analysis

---

**作業ディレクトリ**: `/content/drive/MyDrive/2026/SR_paralog`

### カテゴリー
1. **Chromatin Remodeling** - SWI/SNF, PRC, NuRD, etc.
2. **Metabolism** - MTAP/PRMT5, ENO1/ENO2, etc.
3. **DNA Damage Repair** - BRCA, RAD, PARP, etc.
4. **Splicing Factors** - SF3B, U2AF, etc.
5. **Transcription Factors** - MYC, TEAD, etc.
6. **Kinases** - CDK, MAPK, etc.
7. **Ubiquitin System** - E3 ligases, DUBs, etc.
8. **Cell Cycle** - Cyclins, CDKs, etc.
9. **RNA Processing** - DEAD-box helicases, etc.
10. **Mitochondria** - Respiratory chain, etc.
11. **Cytoskeleton** - Actins, Tubulins, etc.
12. **Signaling** - RTKs, GPCRs, etc.

---
## 1. 環境セットアップ

In [ ]:
# Google Drive マウント
from google.colab import drive
drive.mount('/content/drive')

# 作業ディレクトリ設定
WORK_DIR = '/content/drive/MyDrive/2026/SR_paralog'
DATA_DIR = f'{WORK_DIR}/depmap_25q3'

import os
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(f'{WORK_DIR}/results', exist_ok=True)
os.makedirs(f'{WORK_DIR}/results/by_category', exist_ok=True)

print(f"Work directory: {WORK_DIR}")
print(f"Data directory: {DATA_DIR}")

In [ ]:
# パッケージインストール
!pip install statsmodels -q

import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import false_discovery_control
import statsmodels.api as sm
from typing import List, Dict, Optional
from dataclasses import dataclass
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 150

print("✅ Setup complete!")

---
## 2. 包括的パラログデータベース

In [ ]:
@dataclass
class ParalogPair:
    """パラログペア"""
    gene_a: str           # ドライバー遺伝子
    gene_b: str           # ターゲット遺伝子
    category: str         # 大カテゴリー
    subcategory: str      # サブカテゴリー
    evidence: str         # エビデンスレベル
    pmid: str = ""        # 参考文献


# ============================================================================
# 包括的パラログデータベース
# ============================================================================

PARALOG_DATABASE = [
    # ==========================================================================
    # 1. CHROMATIN REMODELING
    # ==========================================================================
    # SWI/SNF Complex
    ParalogPair("SMARCA4", "SMARCA2", "Chromatin", "SWI/SNF", "published", "26552009"),
    ParalogPair("SMARCA2", "SMARCA4", "Chromatin", "SWI/SNF", "published", "26552009"),
    ParalogPair("ARID1A", "ARID1B", "Chromatin", "SWI/SNF", "published", "24520176"),
    ParalogPair("ARID1B", "ARID1A", "Chromatin", "SWI/SNF", "published", "24520176"),
    ParalogPair("ARID2", "ARID1A", "Chromatin", "SWI/SNF", "predicted", ""),
    ParalogPair("BRD9", "BRD7", "Chromatin", "SWI/SNF", "published", "31253568"),
    ParalogPair("BRD7", "BRD9", "Chromatin", "SWI/SNF", "published", "31253568"),
    ParalogPair("SMARCC1", "SMARCC2", "Chromatin", "SWI/SNF", "predicted", ""),
    ParalogPair("SMARCC2", "SMARCC1", "Chromatin", "SWI/SNF", "predicted", ""),
    ParalogPair("SMARCD1", "SMARCD2", "Chromatin", "SWI/SNF", "predicted", ""),
    ParalogPair("SMARCD2", "SMARCD3", "Chromatin", "SWI/SNF", "predicted", ""),
    ParalogPair("PBRM1", "ARID2", "Chromatin", "SWI/SNF", "published", "29562155"),
    ParalogPair("DPF1", "DPF2", "Chromatin", "SWI/SNF", "predicted", ""),
    ParalogPair("DPF2", "DPF3", "Chromatin", "SWI/SNF", "predicted", ""),
    ParalogPair("BCL7A", "BCL7B", "Chromatin", "SWI/SNF", "predicted", ""),
    ParalogPair("BCL7B", "BCL7C", "Chromatin", "SWI/SNF", "predicted", ""),
    
    # PRC Complex
    ParalogPair("EZH1", "EZH2", "Chromatin", "PRC", "published", "31068699"),
    ParalogPair("EZH2", "EZH1", "Chromatin", "PRC", "published", "31068699"),
    ParalogPair("CBX2", "CBX4", "Chromatin", "PRC", "predicted", ""),
    ParalogPair("CBX4", "CBX8", "Chromatin", "PRC", "predicted", ""),
    ParalogPair("CBX7", "CBX8", "Chromatin", "PRC", "predicted", ""),
    ParalogPair("PCGF1", "PCGF2", "Chromatin", "PRC", "predicted", ""),
    ParalogPair("PCGF2", "PCGF4", "Chromatin", "PRC", "predicted", ""),
    ParalogPair("PCGF4", "PCGF5", "Chromatin", "PRC", "predicted", ""),
    ParalogPair("RING1", "RNF2", "Chromatin", "PRC", "predicted", ""),
    ParalogPair("RNF2", "RING1", "Chromatin", "PRC", "predicted", ""),
    ParalogPair("SUZ12", "EED", "Chromatin", "PRC", "predicted", ""),
    
    # NuRD Complex
    ParalogPair("CHD3", "CHD4", "Chromatin", "NuRD", "predicted", ""),
    ParalogPair("CHD4", "CHD5", "Chromatin", "NuRD", "predicted", ""),
    ParalogPair("MTA1", "MTA2", "Chromatin", "NuRD", "predicted", ""),
    ParalogPair("MTA2", "MTA3", "Chromatin", "NuRD", "predicted", ""),
    ParalogPair("HDAC1", "HDAC2", "Chromatin", "NuRD", "published", ""),
    ParalogPair("HDAC2", "HDAC1", "Chromatin", "NuRD", "published", ""),
    ParalogPair("GATAD2A", "GATAD2B", "Chromatin", "NuRD", "predicted", ""),
    ParalogPair("MBD2", "MBD3", "Chromatin", "NuRD", "predicted", ""),
    
    # HAT/HDAC
    ParalogPair("CREBBP", "EP300", "Chromatin", "HAT", "published", "27571770"),
    ParalogPair("EP300", "CREBBP", "Chromatin", "HAT", "published", "27571770"),
    ParalogPair("KAT2A", "KAT2B", "Chromatin", "HAT", "predicted", ""),
    ParalogPair("KAT6A", "KAT6B", "Chromatin", "HAT", "predicted", ""),
    ParalogPair("HDAC3", "HDAC8", "Chromatin", "HDAC", "predicted", ""),
    
    # Histone Methyltransferases
    ParalogPair("KMT2A", "KMT2B", "Chromatin", "HMT", "predicted", ""),
    ParalogPair("KMT2C", "KMT2D", "Chromatin", "HMT", "published", ""),
    ParalogPair("SETD1A", "SETD1B", "Chromatin", "HMT", "predicted", ""),
    ParalogPair("NSD1", "NSD2", "Chromatin", "HMT", "predicted", ""),
    ParalogPair("NSD2", "NSD3", "Chromatin", "HMT", "predicted", ""),
    
    # Histone Demethylases
    ParalogPair("KDM1A", "KDM1B", "Chromatin", "KDM", "predicted", ""),
    ParalogPair("KDM4A", "KDM4B", "Chromatin", "KDM", "predicted", ""),
    ParalogPair("KDM5A", "KDM5B", "Chromatin", "KDM", "predicted", ""),
    ParalogPair("KDM6A", "KDM6B", "Chromatin", "KDM", "published", ""),
    
    # Cohesin
    ParalogPair("STAG1", "STAG2", "Chromatin", "Cohesin", "published", "33007256"),
    ParalogPair("STAG2", "STAG1", "Chromatin", "Cohesin", "published", "33007256"),
    
    # DNA Methylation
    ParalogPair("DNMT1", "DNMT3A", "Chromatin", "DNA_methylation", "predicted", ""),
    ParalogPair("DNMT3A", "DNMT3B", "Chromatin", "DNA_methylation", "predicted", ""),
    ParalogPair("TET1", "TET2", "Chromatin", "DNA_methylation", "predicted", ""),
    ParalogPair("TET2", "TET3", "Chromatin", "DNA_methylation", "predicted", ""),
    
    # ISWI/INO80
    ParalogPair("SMARCA1", "SMARCA5", "Chromatin", "ISWI", "predicted", ""),
    ParalogPair("SMARCA5", "SMARCA1", "Chromatin", "ISWI", "predicted", ""),
    ParalogPair("BAZ1A", "BAZ1B", "Chromatin", "ISWI", "predicted", ""),
    ParalogPair("BAZ2A", "BAZ2B", "Chromatin", "ISWI", "predicted", ""),
    ParalogPair("INO80", "SRCAP", "Chromatin", "INO80", "predicted", ""),
    
    # ==========================================================================
    # 2. METABOLISM
    # ==========================================================================
    # One-carbon metabolism
    ParalogPair("MTAP", "PRMT5", "Metabolism", "Methionine_salvage", "published", "27416917"),
    ParalogPair("MAT1A", "MAT2A", "Metabolism", "Methionine_salvage", "published", ""),
    ParalogPair("MAT2A", "MAT2B", "Metabolism", "Methionine_salvage", "predicted", ""),
    
    # Glycolysis
    ParalogPair("ENO1", "ENO2", "Metabolism", "Glycolysis", "published", "28178239"),
    ParalogPair("ENO2", "ENO3", "Metabolism", "Glycolysis", "predicted", ""),
    ParalogPair("PKM", "PKLR", "Metabolism", "Glycolysis", "predicted", ""),
    ParalogPair("ALDOA", "ALDOB", "Metabolism", "Glycolysis", "predicted", ""),
    ParalogPair("ALDOB", "ALDOC", "Metabolism", "Glycolysis", "predicted", ""),
    ParalogPair("GAPDH", "GAPDHS", "Metabolism", "Glycolysis", "predicted", ""),
    ParalogPair("HK1", "HK2", "Metabolism", "Glycolysis", "predicted", ""),
    ParalogPair("HK2", "HK3", "Metabolism", "Glycolysis", "predicted", ""),
    ParalogPair("PFKM", "PFKP", "Metabolism", "Glycolysis", "predicted", ""),
    ParalogPair("PFKP", "PFKL", "Metabolism", "Glycolysis", "predicted", ""),
    ParalogPair("LDHA", "LDHB", "Metabolism", "Glycolysis", "predicted", ""),
    ParalogPair("LDHB", "LDHC", "Metabolism", "Glycolysis", "predicted", ""),
    
    # TCA Cycle
    ParalogPair("IDH1", "IDH2", "Metabolism", "TCA_cycle", "predicted", ""),
    ParalogPair("IDH2", "IDH3A", "Metabolism", "TCA_cycle", "predicted", ""),
    ParalogPair("MDH1", "MDH2", "Metabolism", "TCA_cycle", "predicted", ""),
    ParalogPair("FH", "SDHA", "Metabolism", "TCA_cycle", "predicted", ""),
    ParalogPair("CS", "ACLY", "Metabolism", "TCA_cycle", "predicted", ""),
    
    # Glutamine metabolism
    ParalogPair("GLS", "GLS2", "Metabolism", "Glutamine", "predicted", ""),
    ParalogPair("GLUL", "GLUD1", "Metabolism", "Glutamine", "predicted", ""),
    ParalogPair("GLUD1", "GLUD2", "Metabolism", "Glutamine", "predicted", ""),
    ParalogPair("GOT1", "GOT2", "Metabolism", "Glutamine", "predicted", ""),
    
    # Lipid metabolism
    ParalogPair("FASN", "ACACA", "Metabolism", "Lipid", "predicted", ""),
    ParalogPair("ACACA", "ACACB", "Metabolism", "Lipid", "predicted", ""),
    ParalogPair("SCD", "SCD5", "Metabolism", "Lipid", "predicted", ""),
    ParalogPair("HMGCR", "HMGCS1", "Metabolism", "Lipid", "predicted", ""),
    
    # Nucleotide metabolism
    ParalogPair("TYMS", "DHFR", "Metabolism", "Nucleotide", "predicted", ""),
    ParalogPair("RRM1", "RRM2", "Metabolism", "Nucleotide", "predicted", ""),
    ParalogPair("RRM2", "RRM2B", "Metabolism", "Nucleotide", "predicted", ""),
    ParalogPair("DHODH", "UMPS", "Metabolism", "Nucleotide", "predicted", ""),
    ParalogPair("CAD", "DHODH", "Metabolism", "Nucleotide", "predicted", ""),
    ParalogPair("IMPDH1", "IMPDH2", "Metabolism", "Nucleotide", "predicted", ""),
    
    # ==========================================================================
    # 3. DNA DAMAGE REPAIR
    # ==========================================================================
    # Homologous Recombination
    ParalogPair("BRCA1", "BRCA2", "DNA_Repair", "HR", "published", ""),
    ParalogPair("BRCA2", "PALB2", "DNA_Repair", "HR", "published", ""),
    ParalogPair("RAD51", "RAD51B", "DNA_Repair", "HR", "predicted", ""),
    ParalogPair("RAD51B", "RAD51C", "DNA_Repair", "HR", "predicted", ""),
    ParalogPair("RAD51C", "RAD51D", "DNA_Repair", "HR", "predicted", ""),
    ParalogPair("RAD54L", "RAD54B", "DNA_Repair", "HR", "predicted", ""),
    ParalogPair("XRCC2", "XRCC3", "DNA_Repair", "HR", "predicted", ""),
    
    # PARP family
    ParalogPair("PARP1", "PARP2", "DNA_Repair", "PARP", "published", ""),
    ParalogPair("PARP2", "PARP3", "DNA_Repair", "PARP", "predicted", ""),
    
    # Base Excision Repair
    ParalogPair("OGG1", "MUTYH", "DNA_Repair", "BER", "predicted", ""),
    ParalogPair("APEX1", "APEX2", "DNA_Repair", "BER", "predicted", ""),
    ParalogPair("XRCC1", "LIG3", "DNA_Repair", "BER", "predicted", ""),
    ParalogPair("POLB", "POLL", "DNA_Repair", "BER", "predicted", ""),
    
    # Mismatch Repair
    ParalogPair("MLH1", "MLH3", "DNA_Repair", "MMR", "predicted", ""),
    ParalogPair("MSH2", "MSH6", "DNA_Repair", "MMR", "predicted", ""),
    ParalogPair("MSH3", "MSH6", "DNA_Repair", "MMR", "predicted", ""),
    ParalogPair("PMS1", "PMS2", "DNA_Repair", "MMR", "predicted", ""),
    
    # Non-homologous End Joining
    ParalogPair("XRCC4", "XRCC5", "DNA_Repair", "NHEJ", "predicted", ""),
    ParalogPair("XRCC5", "XRCC6", "DNA_Repair", "NHEJ", "predicted", ""),
    ParalogPair("LIG4", "LIG3", "DNA_Repair", "NHEJ", "predicted", ""),
    ParalogPair("DCLRE1C", "PRKDC", "DNA_Repair", "NHEJ", "predicted", ""),
    
    # Fanconi Anemia
    ParalogPair("FANCA", "FANCC", "DNA_Repair", "FA", "predicted", ""),
    ParalogPair("FANCD2", "FANCI", "DNA_Repair", "FA", "published", ""),
    ParalogPair("FANCM", "FANCJ", "DNA_Repair", "FA", "predicted", ""),
    
    # ATM/ATR pathway
    ParalogPair("ATM", "ATR", "DNA_Repair", "Checkpoint", "published", ""),
    ParalogPair("CHEK1", "CHEK2", "DNA_Repair", "Checkpoint", "published", ""),
    
    # ==========================================================================
    # 4. SPLICING FACTORS
    # ==========================================================================
    ParalogPair("SF3B1", "SF3B2", "Splicing", "SF3B", "predicted", ""),
    ParalogPair("SF3B2", "SF3B3", "Splicing", "SF3B", "predicted", ""),
    ParalogPair("U2AF1", "U2AF2", "Splicing", "U2AF", "published", ""),
    ParalogPair("SRSF1", "SRSF2", "Splicing", "SRSF", "predicted", ""),
    ParalogPair("SRSF2", "SRSF3", "Splicing", "SRSF", "predicted", ""),
    ParalogPair("SRSF3", "SRSF6", "Splicing", "SRSF", "predicted", ""),
    ParalogPair("HNRNPA1", "HNRNPA2B1", "Splicing", "hnRNP", "predicted", ""),
    ParalogPair("HNRNPC", "HNRNPD", "Splicing", "hnRNP", "predicted", ""),
    ParalogPair("HNRNPH1", "HNRNPH2", "Splicing", "hnRNP", "predicted", ""),
    ParalogPair("HNRNPK", "HNRNPL", "Splicing", "hnRNP", "predicted", ""),
    ParalogPair("PRPF8", "PRPF31", "Splicing", "PRP", "predicted", ""),
    ParalogPair("SNRPA", "SNRPB", "Splicing", "snRNP", "predicted", ""),
    ParalogPair("SNRPD1", "SNRPD2", "Splicing", "snRNP", "predicted", ""),
    ParalogPair("SNRPD2", "SNRPD3", "Splicing", "snRNP", "predicted", ""),
    ParalogPair("RBM10", "RBM5", "Splicing", "RBM", "predicted", ""),
    ParalogPair("RBMX", "RBMXL1", "Splicing", "RBM", "predicted", ""),
    
    # ==========================================================================
    # 5. TRANSCRIPTION FACTORS
    # ==========================================================================
    ParalogPair("MYC", "MYCN", "Transcription", "MYC", "published", ""),
    ParalogPair("MYCN", "MYCL", "Transcription", "MYC", "predicted", ""),
    ParalogPair("MAX", "MXD1", "Transcription", "MYC", "predicted", ""),
    ParalogPair("TEAD1", "TEAD2", "Transcription", "TEAD", "predicted", ""),
    ParalogPair("TEAD2", "TEAD3", "Transcription", "TEAD", "predicted", ""),
    ParalogPair("TEAD3", "TEAD4", "Transcription", "TEAD", "predicted", ""),
    ParalogPair("YAP1", "WWTR1", "Transcription", "Hippo", "published", ""),
    ParalogPair("SMAD2", "SMAD3", "Transcription", "SMAD", "predicted", ""),
    ParalogPair("SMAD3", "SMAD4", "Transcription", "SMAD", "predicted", ""),
    ParalogPair("RUNX1", "RUNX2", "Transcription", "RUNX", "predicted", ""),
    ParalogPair("RUNX2", "RUNX3", "Transcription", "RUNX", "predicted", ""),
    ParalogPair("GATA1", "GATA2", "Transcription", "GATA", "predicted", ""),
    ParalogPair("GATA2", "GATA3", "Transcription", "GATA", "predicted", ""),
    ParalogPair("GATA3", "GATA4", "Transcription", "GATA", "predicted", ""),
    ParalogPair("FOXA1", "FOXA2", "Transcription", "FOX", "predicted", ""),
    ParalogPair("FOXM1", "FOXO1", "Transcription", "FOX", "predicted", ""),
    ParalogPair("FOXO1", "FOXO3", "Transcription", "FOX", "predicted", ""),
    ParalogPair("TP53", "TP63", "Transcription", "p53_family", "published", ""),
    ParalogPair("TP63", "TP73", "Transcription", "p53_family", "predicted", ""),
    ParalogPair("RB1", "RBL1", "Transcription", "RB_family", "predicted", ""),
    ParalogPair("RBL1", "RBL2", "Transcription", "RB_family", "predicted", ""),
    ParalogPair("E2F1", "E2F2", "Transcription", "E2F", "predicted", ""),
    ParalogPair("E2F2", "E2F3", "Transcription", "E2F", "predicted", ""),
    ParalogPair("E2F3", "E2F4", "Transcription", "E2F", "predicted", ""),
    ParalogPair("NFE2L2", "NFE2L1", "Transcription", "NRF", "predicted", ""),
    ParalogPair("HIF1A", "EPAS1", "Transcription", "HIF", "predicted", ""),
    ParalogPair("STAT1", "STAT2", "Transcription", "STAT", "predicted", ""),
    ParalogPair("STAT3", "STAT5A", "Transcription", "STAT", "predicted", ""),
    ParalogPair("STAT5A", "STAT5B", "Transcription", "STAT", "predicted", ""),
    
    # ==========================================================================
    # 6. KINASES
    # ==========================================================================
    # CDKs
    ParalogPair("CDK1", "CDK2", "Kinase", "CDK", "predicted", ""),
    ParalogPair("CDK2", "CDK4", "Kinase", "CDK", "predicted", ""),
    ParalogPair("CDK4", "CDK6", "Kinase", "CDK", "published", ""),
    ParalogPair("CDK7", "CDK9", "Kinase", "CDK", "predicted", ""),
    ParalogPair("CDK12", "CDK13", "Kinase", "CDK", "predicted", ""),
    
    # MAPK pathway
    ParalogPair("MAPK1", "MAPK3", "Kinase", "MAPK", "predicted", ""),
    ParalogPair("MAP2K1", "MAP2K2", "Kinase", "MAPK", "predicted", ""),
    ParalogPair("MAP3K1", "MAP3K2", "Kinase", "MAPK", "predicted", ""),
    ParalogPair("BRAF", "RAF1", "Kinase", "RAF", "published", ""),
    ParalogPair("RAF1", "ARAF", "Kinase", "RAF", "predicted", ""),
    
    # PI3K pathway
    ParalogPair("PIK3CA", "PIK3CB", "Kinase", "PI3K", "predicted", ""),
    ParalogPair("PIK3CB", "PIK3CD", "Kinase", "PI3K", "predicted", ""),
    ParalogPair("AKT1", "AKT2", "Kinase", "AKT", "predicted", ""),
    ParalogPair("AKT2", "AKT3", "Kinase", "AKT", "predicted", ""),
    ParalogPair("MTOR", "RPTOR", "Kinase", "mTOR", "predicted", ""),
    ParalogPair("TSC1", "TSC2", "Kinase", "mTOR", "predicted", ""),
    
    # Receptor Tyrosine Kinases
    ParalogPair("EGFR", "ERBB2", "Kinase", "RTK", "predicted", ""),
    ParalogPair("ERBB2", "ERBB3", "Kinase", "RTK", "published", ""),
    ParalogPair("ERBB3", "ERBB4", "Kinase", "RTK", "predicted", ""),
    ParalogPair("FGFR1", "FGFR2", "Kinase", "RTK", "predicted", ""),
    ParalogPair("FGFR2", "FGFR3", "Kinase", "RTK", "predicted", ""),
    ParalogPair("FGFR3", "FGFR4", "Kinase", "RTK", "predicted", ""),
    ParalogPair("MET", "MST1R", "Kinase", "RTK", "predicted", ""),
    ParalogPair("IGF1R", "INSR", "Kinase", "RTK", "predicted", ""),
    
    # Aurora kinases
    ParalogPair("AURKA", "AURKB", "Kinase", "Aurora", "published", ""),
    ParalogPair("AURKB", "AURKC", "Kinase", "Aurora", "predicted", ""),
    
    # PLK
    ParalogPair("PLK1", "PLK2", "Kinase", "PLK", "predicted", ""),
    ParalogPair("PLK2", "PLK3", "Kinase", "PLK", "predicted", ""),
    ParalogPair("PLK3", "PLK4", "Kinase", "PLK", "predicted", ""),
    
    # SRC family
    ParalogPair("SRC", "YES1", "Kinase", "SRC", "predicted", ""),
    ParalogPair("YES1", "FYN", "Kinase", "SRC", "predicted", ""),
    ParalogPair("LYN", "HCK", "Kinase", "SRC", "predicted", ""),
    
    # ==========================================================================
    # 7. UBIQUITIN SYSTEM
    # ==========================================================================
    # E3 ligases
    ParalogPair("MDM2", "MDM4", "Ubiquitin", "E3_ligase", "published", ""),
    ParalogPair("FBXW7", "FBXW11", "Ubiquitin", "E3_ligase", "predicted", ""),
    ParalogPair("CUL1", "CUL2", "Ubiquitin", "Cullin", "predicted", ""),
    ParalogPair("CUL2", "CUL3", "Ubiquitin", "Cullin", "predicted", ""),
    ParalogPair("CUL3", "CUL4A", "Ubiquitin", "Cullin", "predicted", ""),
    ParalogPair("CUL4A", "CUL4B", "Ubiquitin", "Cullin", "predicted", ""),
    ParalogPair("BIRC2", "BIRC3", "Ubiquitin", "E3_ligase", "predicted", ""),
    ParalogPair("TRIM24", "TRIM28", "Ubiquitin", "TRIM", "predicted", ""),
    ParalogPair("TRIM28", "TRIM33", "Ubiquitin", "TRIM", "predicted", ""),
    
    # DUBs
    ParalogPair("USP1", "USP7", "Ubiquitin", "DUB", "predicted", ""),
    ParalogPair("USP7", "USP8", "Ubiquitin", "DUB", "predicted", ""),
    ParalogPair("USP14", "USP15", "Ubiquitin", "DUB", "predicted", ""),
    ParalogPair("USP28", "USP25", "Ubiquitin", "DUB", "predicted", ""),
    
    # ==========================================================================
    # 8. CELL CYCLE
    # ==========================================================================
    # Cyclins
    ParalogPair("CCNA1", "CCNA2", "Cell_Cycle", "Cyclin", "predicted", ""),
    ParalogPair("CCNB1", "CCNB2", "Cell_Cycle", "Cyclin", "predicted", ""),
    ParalogPair("CCND1", "CCND2", "Cell_Cycle", "Cyclin", "predicted", ""),
    ParalogPair("CCND2", "CCND3", "Cell_Cycle", "Cyclin", "predicted", ""),
    ParalogPair("CCNE1", "CCNE2", "Cell_Cycle", "Cyclin", "predicted", ""),
    
    # CDK inhibitors
    ParalogPair("CDKN1A", "CDKN1B", "Cell_Cycle", "CDKI", "predicted", ""),
    ParalogPair("CDKN1B", "CDKN1C", "Cell_Cycle", "CDKI", "predicted", ""),
    ParalogPair("CDKN2A", "CDKN2B", "Cell_Cycle", "CDKI", "published", ""),
    ParalogPair("CDKN2B", "CDKN2C", "Cell_Cycle", "CDKI", "predicted", ""),
    ParalogPair("CDKN2C", "CDKN2D", "Cell_Cycle", "CDKI", "predicted", ""),
    
    # Mitotic checkpoint
    ParalogPair("BUB1", "BUB1B", "Cell_Cycle", "SAC", "predicted", ""),
    ParalogPair("MAD1L1", "MAD2L1", "Cell_Cycle", "SAC", "predicted", ""),
    ParalogPair("CDC20", "FZR1", "Cell_Cycle", "APC", "predicted", ""),
    ParalogPair("ANAPC1", "ANAPC2", "Cell_Cycle", "APC", "predicted", ""),
    
    # ==========================================================================
    # 9. RNA PROCESSING
    # ==========================================================================
    # DEAD-box helicases
    ParalogPair("DDX3X", "DDX3Y", "RNA_Processing", "DDX", "predicted", ""),
    ParalogPair("DDX5", "DDX17", "RNA_Processing", "DDX", "predicted", ""),
    ParalogPair("DDX41", "DDX46", "RNA_Processing", "DDX", "predicted", ""),
    ParalogPair("DHX9", "DHX36", "RNA_Processing", "DDX", "predicted", ""),
    
    # RNA polymerase
    ParalogPair("POLR2A", "POLR2B", "RNA_Processing", "RNAP", "predicted", ""),
    
    # Translation initiation
    ParalogPair("EIF4A1", "EIF4A2", "RNA_Processing", "Translation", "predicted", ""),
    ParalogPair("EIF4E", "EIF4EBP1", "RNA_Processing", "Translation", "predicted", ""),
    ParalogPair("EIF2S1", "EIF2S2", "RNA_Processing", "Translation", "predicted", ""),
    ParalogPair("EIF3A", "EIF3B", "RNA_Processing", "Translation", "predicted", ""),
    
    # Ribosome
    ParalogPair("RPL5", "RPL11", "RNA_Processing", "Ribosome", "predicted", ""),
    ParalogPair("RPS6", "RPS14", "RNA_Processing", "Ribosome", "predicted", ""),
    
    # ==========================================================================
    # 10. MITOCHONDRIA
    # ==========================================================================
    # Respiratory chain
    ParalogPair("NDUFA1", "NDUFA2", "Mitochondria", "Complex_I", "predicted", ""),
    ParalogPair("NDUFS1", "NDUFS2", "Mitochondria", "Complex_I", "predicted", ""),
    ParalogPair("UQCRC1", "UQCRC2", "Mitochondria", "Complex_III", "predicted", ""),
    ParalogPair("COX4I1", "COX4I2", "Mitochondria", "Complex_IV", "predicted", ""),
    ParalogPair("COX5A", "COX5B", "Mitochondria", "Complex_IV", "predicted", ""),
    ParalogPair("ATP5F1A", "ATP5F1B", "Mitochondria", "Complex_V", "predicted", ""),
    
    # Mitochondrial dynamics
    ParalogPair("MFN1", "MFN2", "Mitochondria", "Fusion", "predicted", ""),
    ParalogPair("DRP1", "OPA1", "Mitochondria", "Fission", "predicted", ""),
    
    # Apoptosis
    ParalogPair("BCL2", "BCL2L1", "Mitochondria", "Apoptosis", "published", ""),
    ParalogPair("BCL2L1", "MCL1", "Mitochondria", "Apoptosis", "published", ""),
    ParalogPair("BAX", "BAK1", "Mitochondria", "Apoptosis", "published", ""),
    ParalogPair("BID", "BIM", "Mitochondria", "Apoptosis", "predicted", ""),
    
    # ==========================================================================
    # 11. CYTOSKELETON
    # ==========================================================================
    # Actin
    ParalogPair("ACTB", "ACTG1", "Cytoskeleton", "Actin", "predicted", ""),
    ParalogPair("ACTA1", "ACTA2", "Cytoskeleton", "Actin", "predicted", ""),
    
    # Tubulin
    ParalogPair("TUBA1A", "TUBA1B", "Cytoskeleton", "Tubulin", "predicted", ""),
    ParalogPair("TUBB", "TUBB4A", "Cytoskeleton", "Tubulin", "predicted", ""),
    ParalogPair("TUBB4A", "TUBB4B", "Cytoskeleton", "Tubulin", "predicted", ""),
    
    # Motor proteins
    ParalogPair("KIF2A", "KIF2C", "Cytoskeleton", "Kinesin", "predicted", ""),
    ParalogPair("KIF11", "KIF15", "Cytoskeleton", "Kinesin", "predicted", ""),
    ParalogPair("DYNC1H1", "DYNC2H1", "Cytoskeleton", "Dynein", "predicted", ""),
    
    # ==========================================================================
    # 12. SIGNALING
    # ==========================================================================
    # Wnt pathway
    ParalogPair("CTNNB1", "JUP", "Signaling", "Wnt", "predicted", ""),
    ParalogPair("APC", "APC2", "Signaling", "Wnt", "predicted", ""),
    ParalogPair("GSK3A", "GSK3B", "Signaling", "Wnt", "predicted", ""),
    ParalogPair("DVL1", "DVL2", "Signaling", "Wnt", "predicted", ""),
    ParalogPair("DVL2", "DVL3", "Signaling", "Wnt", "predicted", ""),
    
    # Notch pathway
    ParalogPair("NOTCH1", "NOTCH2", "Signaling", "Notch", "predicted", ""),
    ParalogPair("NOTCH2", "NOTCH3", "Signaling", "Notch", "predicted", ""),
    ParalogPair("NOTCH3", "NOTCH4", "Signaling", "Notch", "predicted", ""),
    ParalogPair("JAG1", "JAG2", "Signaling", "Notch", "predicted", ""),
    ParalogPair("DLL1", "DLL4", "Signaling", "Notch", "predicted", ""),
    
    # Hedgehog pathway
    ParalogPair("GLI1", "GLI2", "Signaling", "Hedgehog", "predicted", ""),
    ParalogPair("GLI2", "GLI3", "Signaling", "Hedgehog", "predicted", ""),
    
    # NF-kB pathway
    ParalogPair("RELA", "RELB", "Signaling", "NFkB", "predicted", ""),
    ParalogPair("NFKB1", "NFKB2", "Signaling", "NFkB", "predicted", ""),
    ParalogPair("IKBKA", "IKBKB", "Signaling", "NFkB", "predicted", ""),
    
    # RAS family
    ParalogPair("KRAS", "NRAS", "Signaling", "RAS", "published", ""),
    ParalogPair("NRAS", "HRAS", "Signaling", "RAS", "predicted", ""),
    ParalogPair("RAC1", "RAC2", "Signaling", "RAS", "predicted", ""),
    ParalogPair("RHOA", "RHOB", "Signaling", "RAS", "predicted", ""),
    ParalogPair("RHOB", "RHOC", "Signaling", "RAS", "predicted", ""),
]

print(f"Total paralog pairs in database: {len(PARALOG_DATABASE)}")

# カテゴリー集計
category_df = pd.DataFrame([(p.category, p.subcategory) for p in PARALOG_DATABASE], 
                           columns=['Category', 'Subcategory'])
print("\n📊 Pairs by Category:")
display(category_df['Category'].value_counts().to_frame('count'))

print("\n📊 Pairs by Subcategory (top 20):")
display(category_df['Subcategory'].value_counts().head(20).to_frame('count'))

---
## 3. データローダー & 解析クラス

In [ ]:
class DepMap25Q3Loader:
    """DepMap Public 25Q3 データローダー"""
    
    FILE_MAPPING = {
        'crispr': 'CRISPRGeneEffect.csv',
        'model': 'Model.csv', 
        'mutations': 'OmicsSomaticMutations.csv',
        'expression': 'OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv',
        'cnv': 'OmicsCNGene.csv',
    }
    
    def __init__(self, data_dir: str):
        self.data_dir = Path(data_dir)
        self.crispr = None
        self.model = None
        self.mutations = None
        self.expression = None
        self.cnv = None
        
    def load_all(self):
        print("="*60)
        print("Loading DepMap 25Q3 data...")
        print("="*60)
        
        # CRISPR
        print("\n📊 Loading CRISPR...")
        self.crispr = pd.read_csv(self.data_dir / self.FILE_MAPPING['crispr'], index_col=0)
        self.crispr.columns = [c.split(' ')[0] for c in self.crispr.columns]
        print(f"   ✅ {self.crispr.shape[0]} × {self.crispr.shape[1]}")
        
        # Model
        print("\n📋 Loading Model...")
        self.model = pd.read_csv(self.data_dir / self.FILE_MAPPING['model']).set_index('ModelID')
        print(f"   ✅ {len(self.model)} cell lines")
        
        # Mutations
        print("\n🧬 Loading Mutations...")
        self.mutations = pd.read_csv(self.data_dir / self.FILE_MAPPING['mutations'], low_memory=False)
        print(f"   ✅ {len(self.mutations)} mutations")
        
        # Expression
        print("\n📈 Loading Expression...")
        expr_raw = pd.read_csv(self.data_dir / self.FILE_MAPPING['expression'])
        self.expression = expr_raw.set_index('ModelID')
        meta_cols = ['Unnamed: 0', 'SequencingID', 'IsDefaultEntryForModel', 'ModelConditionID', 'IsDefaultEntryForMC']
        self.expression = self.expression.drop(columns=[c for c in meta_cols if c in self.expression.columns])
        self.expression.columns = [c.split(' ')[0] for c in self.expression.columns]
        if self.expression.index.duplicated().any():
            self.expression = self.expression[~self.expression.index.duplicated(keep='first')]
        print(f"   ✅ {self.expression.shape[0]} × {self.expression.shape[1]}")
        
        # CNV (optional)
        cnv_path = self.data_dir / self.FILE_MAPPING['cnv']
        if cnv_path.exists():
            print("\n📉 Loading CNV...")
            self.cnv = pd.read_csv(cnv_path, index_col=0)
            self.cnv.columns = [c.split(' ')[0] for c in self.cnv.columns]
            print(f"   ✅ {self.cnv.shape[0]} × {self.cnv.shape[1]}")
        
        print("\n" + "="*60)
        return self

In [ ]:
class ConfounderAdjustedAnalyzer:
    """交絡補正付き解析クラス"""
    
    def __init__(self, loader):
        self.loader = loader
        
    def get_lof_status(self, gene):
        if self.loader.mutations is None:
            return pd.Series(dtype=bool)
        
        gene_muts = self.loader.mutations[self.loader.mutations['HugoSymbol'] == gene].copy()
        if len(gene_muts) == 0:
            return pd.Series([False] * len(self.loader.crispr), index=self.loader.crispr.index)
        
        lof_mask = pd.Series([False] * len(gene_muts), index=gene_muts.index)
        
        if 'VariantType' in gene_muts.columns and 'Ref' in gene_muts.columns:
            lof_mask |= ((gene_muts['VariantType'].isin(['deletion', 'insertion'])) &
                        (gene_muts['Ref'].str.len() != gene_muts['Alt'].str.len()))
        
        if 'ProveanPrediction' in gene_muts.columns:
            lof_mask |= (gene_muts['ProveanPrediction'] == 'Damaging')
        if 'AMClass' in gene_muts.columns:
            lof_mask |= gene_muts['AMClass'].str.contains('pathogenic', case=False, na=False)
        if 'AMPathogenicity' in gene_muts.columns:
            lof_mask |= (gene_muts['AMPathogenicity'].fillna(0) > 0.5)
        
        lof_lines = set(gene_muts[lof_mask]['ModelID'].unique())
        return pd.Series([line in lof_lines for line in self.loader.crispr.index], 
                        index=self.loader.crispr.index)
    
    def get_low_expression_status(self, gene, percentile=25):
        if self.loader.expression is None or gene not in self.loader.expression.columns:
            return pd.Series([False] * len(self.loader.crispr), index=self.loader.crispr.index)
        expr = self.loader.expression[gene]
        threshold = np.nanpercentile(expr.dropna(), percentile)
        return pd.Series(expr <= threshold)
    
    def run_ols_analysis(self, driver, target, stratify_by="any", min_affected=5):
        if target not in self.loader.crispr.columns:
            return None
        
        data = pd.DataFrame({'dependency': self.loader.crispr[target]})
        
        if stratify_by == "any":
            lof = self.get_lof_status(driver)
            low_expr = self.get_low_expression_status(driver)
            data['driver_status'] = lof.fillna(False) | low_expr.fillna(False)
        elif stratify_by == "mutation":
            data['driver_status'] = self.get_lof_status(driver)
        else:
            data['driver_status'] = self.get_low_expression_status(driver)
        
        # Lineage
        if self.loader.model is not None and 'OncotreeLineage' in self.loader.model.columns:
            lineage = self.loader.model['OncotreeLineage'].reindex(data.index)
            for lin in lineage.value_counts().head(10).index:
                safe_name = f'lineage_{re.sub(r"[^a-zA-Z0-9]", "_", str(lin))}'
                data[safe_name] = (lineage == lin).astype(int)
        
        if self.loader.expression is not None and target in self.loader.expression.columns:
            data['target_expr'] = self.loader.expression[target].reindex(data.index)
        
        data = data.dropna()
        if len(data) < 20:
            return None
        
        n_affected = data['driver_status'].sum()
        if n_affected < min_affected or (len(data) - n_affected) < min_affected:
            return None
        
        covariates = [c for c in data.columns if c.startswith('lineage_') or c == 'target_expr']
        
        try:
            y = data['dependency'].values
            X = sm.add_constant(data[['driver_status'] + covariates].astype(float).values)
            model = sm.OLS(y, X).fit()
            
            return {
                'driver_gene': driver,
                'target_gene': target,
                'n_affected': int(n_affected),
                'n_unaffected': int(len(data) - n_affected),
                'raw_delta': data[data['driver_status']]['dependency'].mean() - data[~data['driver_status']]['dependency'].mean(),
                'adjusted_coef': model.params[1],
                'adjusted_pvalue': model.pvalues[1],
                'ci_lower': model.conf_int()[1, 0],
                'ci_upper': model.conf_int()[1, 1],
                'r_squared': model.rsquared,
            }
        except:
            return None

In [ ]:
class ComprehensiveParalogScanner:
    """全カテゴリー対応パラログスキャナー"""
    
    def __init__(self, loader):
        self.loader = loader
        self.analyzer = ConfounderAdjustedAnalyzer(loader)
        self.results = pd.DataFrame()
        self.results_by_category = {}
    
    def scan_all(self, pairs=None, stratify_by="any", min_affected=5, verbose=True):
        if pairs is None:
            pairs = PARALOG_DATABASE
        
        print(f"\n{'='*60}")
        print(f"Scanning {len(pairs)} paralog pairs...")
        print(f"{'='*60}")
        
        results = []
        for i, pair in enumerate(pairs):
            if verbose and (i + 1) % 20 == 0:
                print(f"  Progress: {i+1}/{len(pairs)}")
            
            result = self.analyzer.run_ols_analysis(pair.gene_a, pair.gene_b, stratify_by, min_affected)
            if result:
                result['category'] = pair.category
                result['subcategory'] = pair.subcategory
                result['evidence'] = pair.evidence
                result['pmid'] = pair.pmid
                results.append(result)
        
        df = pd.DataFrame(results)
        if len(df) > 0:
            df['fdr'] = false_discovery_control(df['adjusted_pvalue'], method='bh')
            df = df.sort_values('adjusted_pvalue')
            df['significant_fdr05'] = df['fdr'] < 0.05
            df['significant_fdr10'] = df['fdr'] < 0.10
        
        self.results = df
        
        # カテゴリー別に分割
        for cat in df['category'].unique():
            self.results_by_category[cat] = df[df['category'] == cat].copy()
        
        print(f"\n✅ Complete! {len(df)} pairs analyzed.")
        return df
    
    def scan_category(self, category, stratify_by="any", min_affected=5):
        """特定カテゴリーのみスキャン"""
        pairs = [p for p in PARALOG_DATABASE if p.category == category]
        return self.scan_all(pairs, stratify_by, min_affected)
    
    def get_summary(self):
        """カテゴリー別サマリー"""
        if len(self.results) == 0:
            return pd.DataFrame()
        
        return self.results.groupby('category').agg({
            'driver_gene': 'count',
            'significant_fdr05': 'sum',
            'significant_fdr10': 'sum',
            'adjusted_coef': 'mean',
            'adjusted_pvalue': 'min',
        }).rename(columns={
            'driver_gene': 'n_pairs',
            'significant_fdr05': 'sig_fdr05',
            'significant_fdr10': 'sig_fdr10',
            'adjusted_coef': 'mean_effect',
            'adjusted_pvalue': 'best_pval',
        }).sort_values('sig_fdr10', ascending=False)

---
## 4. 可視化関数

In [ ]:
def plot_volcano_by_category(results, save_dir=None):
    """カテゴリー別Volcano plot"""
    categories = results['category'].unique()
    n_cat = len(categories)
    n_cols = 3
    n_rows = (n_cat + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4*n_rows))
    axes = axes.flatten()
    
    for i, cat in enumerate(categories):
        ax = axes[i]
        df = results[results['category'] == cat].copy()
        df['neg_log_p'] = -np.log10(df['adjusted_pvalue'])
        
        colors = ['red' if (fdr < 0.05 and c < 0) else 'orange' if (fdr < 0.1 and c < 0) else 'lightgray'
                  for fdr, c in zip(df['fdr'], df['adjusted_coef'])]
        
        ax.scatter(df['adjusted_coef'], df['neg_log_p'], c=colors, alpha=0.6, s=50)
        ax.axhline(-np.log10(0.05), color='blue', linestyle='--', alpha=0.3)
        ax.axvline(0, color='gray', linestyle='-', alpha=0.3)
        ax.set_title(f"{cat}\n(n={len(df)}, sig={df['significant_fdr10'].sum()})")
        ax.set_xlabel('Adjusted Coef')
        ax.set_ylabel('-log10(p)')
        
        # Top hits ラベル
        for _, row in df[df['fdr'] < 0.1].nsmallest(3, 'adjusted_pvalue').iterrows():
            ax.annotate(f"{row['driver_gene']}→{row['target_gene']}",
                       (row['adjusted_coef'], row['neg_log_p']), fontsize=6)
    
    # 余白を非表示
    for j in range(i+1, len(axes)):
        axes[j].set_visible(False)
    
    plt.tight_layout()
    if save_dir:
        plt.savefig(f"{save_dir}/volcano_by_category.png", dpi=150, bbox_inches='tight')
        print(f"Saved: {save_dir}/volcano_by_category.png")
    plt.show()


def plot_category_summary(results, save_dir=None):
    """カテゴリー別サマリーバーチャート"""
    summary = results.groupby('category').agg({
        'driver_gene': 'count',
        'significant_fdr05': 'sum',
        'significant_fdr10': 'sum',
    }).rename(columns={'driver_gene': 'total'})
    summary = summary.sort_values('significant_fdr10', ascending=True)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    
    y = range(len(summary))
    ax.barh(y, summary['total'], color='lightgray', label='Total tested')
    ax.barh(y, summary['significant_fdr10'], color='orange', label='FDR < 0.10')
    ax.barh(y, summary['significant_fdr05'], color='red', label='FDR < 0.05')
    
    ax.set_yticks(y)
    ax.set_yticklabels(summary.index)
    ax.set_xlabel('Number of paralog pairs')
    ax.set_title('Synthetic Lethality Hits by Category')
    ax.legend(loc='lower right')
    
    plt.tight_layout()
    if save_dir:
        plt.savefig(f"{save_dir}/category_summary.png", dpi=150, bbox_inches='tight')
        print(f"Saved: {save_dir}/category_summary.png")
    plt.show()


def plot_top_hits_forest(results, n_top=30, save_dir=None):
    """全カテゴリーのTop hits Forest plot"""
    df = results.nsmallest(n_top, 'adjusted_pvalue').sort_values('adjusted_coef')
    
    fig, ax = plt.subplots(figsize=(12, max(8, n_top*0.35)))
    y = range(len(df))
    
    xerr = np.array([df['adjusted_coef'] - df['ci_lower'], df['ci_upper'] - df['adjusted_coef']])
    
    # カテゴリーごとに色分け
    categories = df['category'].unique()
    cmap = plt.cm.get_cmap('tab10', len(categories))
    cat_colors = {cat: cmap(i) for i, cat in enumerate(categories)}
    colors = [cat_colors[c] for c in df['category']]
    
    ax.errorbar(df['adjusted_coef'], y, xerr=xerr, fmt='none', ecolor='gray', capsize=2)
    ax.scatter(df['adjusted_coef'], y, c=colors, s=80, zorder=5, edgecolors='black', linewidths=0.5)
    ax.axvline(0, color='black', linestyle='-', alpha=0.3)
    
    labels = [f"{r['driver_gene']}→{r['target_gene']} [{r['category'][:8]}]" for _, r in df.iterrows()]
    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=8)
    ax.set_xlabel('Adjusted Coefficient (95% CI)')
    ax.set_title(f'Top {n_top} Synthetic Lethality Hits (All Categories)')
    
    plt.tight_layout()
    if save_dir:
        plt.savefig(f"{save_dir}/top_hits_forest.png", dpi=150, bbox_inches='tight')
        print(f"Saved: {save_dir}/top_hits_forest.png")
    plt.show()

---
## 5. 🚀 解析実行

In [ ]:
# データ読み込み
loader = DepMap25Q3Loader(DATA_DIR)
loader.load_all()

In [ ]:
# 全カテゴリースキャン
scanner = ComprehensiveParalogScanner(loader)
results = scanner.scan_all(stratify_by="any", min_affected=5, verbose=True)

print(f"\n{'='*60}")
print("OVERALL SUMMARY")
print(f"{'='*60}")
print(f"Total pairs tested: {len(results)}")
print(f"Significant (FDR < 0.05): {results['significant_fdr05'].sum()}")
print(f"Significant (FDR < 0.10): {results['significant_fdr10'].sum()}")

In [ ]:
# カテゴリー別サマリー
print("\n" + "="*70)
print("SUMMARY BY CATEGORY")
print("="*70)
display(scanner.get_summary())

In [ ]:
# Top 30 全体
print("\n" + "="*70)
print("TOP 30 HITS (ALL CATEGORIES)")
print("="*70)
display(results.head(30)[['driver_gene', 'target_gene', 'category', 'subcategory', 
                          'n_affected', 'adjusted_coef', 'adjusted_pvalue', 'fdr', 'evidence']])

---
## 6. 可視化

In [ ]:
SAVE_DIR = f"{WORK_DIR}/results/by_category"

# カテゴリーサマリー
plot_category_summary(results, save_dir=SAVE_DIR)

In [ ]:
# カテゴリー別Volcano
plot_volcano_by_category(results, save_dir=SAVE_DIR)

In [ ]:
# Top hits Forest plot
plot_top_hits_forest(results, n_top=30, save_dir=SAVE_DIR)

---
## 7. カテゴリー別に結果保存

In [ ]:
# 全結果保存
results.to_csv(f"{WORK_DIR}/results/all_paralog_scan_results.csv", index=False)

# FDR < 0.1 のみ
sig_results = results[results['fdr'] < 0.1]
sig_results.to_csv(f"{WORK_DIR}/results/significant_hits_fdr10_all.csv", index=False)

# カテゴリー別に保存
for cat, cat_df in scanner.results_by_category.items():
    safe_cat = re.sub(r'[^a-zA-Z0-9]', '_', cat)
    cat_df.to_csv(f"{SAVE_DIR}/{safe_cat}_results.csv", index=False)
    
    # 有意なもののみも保存
    sig_cat = cat_df[cat_df['fdr'] < 0.1]
    if len(sig_cat) > 0:
        sig_cat.to_csv(f"{SAVE_DIR}/{safe_cat}_significant.csv", index=False)

print(f"\n✅ Results saved!")
print(f"   - {WORK_DIR}/results/all_paralog_scan_results.csv ({len(results)} pairs)")
print(f"   - {WORK_DIR}/results/significant_hits_fdr10_all.csv ({len(sig_results)} hits)")
print(f"   - Category-specific files in {SAVE_DIR}/")

---
## 8. カテゴリー別詳細表示

In [ ]:
# 各カテゴリーのTop hits を表示
for cat in sorted(results['category'].unique()):
    cat_df = results[results['category'] == cat]
    sig_count = cat_df['significant_fdr10'].sum()
    
    print(f"\n{'='*70}")
    print(f"📂 {cat} (Tested: {len(cat_df)}, Significant FDR<0.1: {sig_count})")
    print(f"{'='*70}")
    
    if sig_count > 0:
        display(cat_df[cat_df['fdr'] < 0.1][['driver_gene', 'target_gene', 'subcategory', 
                                              'n_affected', 'adjusted_coef', 'fdr', 'evidence']])
    else:
        print("   No significant hits at FDR < 0.1")
        print("   Top 3 pairs:")
        display(cat_df.head(3)[['driver_gene', 'target_gene', 'subcategory', 
                                'n_affected', 'adjusted_coef', 'adjusted_pvalue']])

---
## 9. まとめ

### 解析カテゴリー
| カテゴリー | 内容 |
|-----------|------|
| Chromatin | SWI/SNF, PRC, NuRD, HAT, HMT, KDM, Cohesin |
| Metabolism | Glycolysis, TCA, Glutamine, Lipid, Nucleotide |
| DNA_Repair | HR, PARP, BER, MMR, NHEJ, FA, Checkpoint |
| Splicing | SF3B, U2AF, SRSF, hnRNP, snRNP |
| Transcription | MYC, TEAD, SMAD, RUNX, GATA, FOX, p53, RB, E2F |
| Kinase | CDK, MAPK, PI3K, RTK, Aurora, PLK, SRC |
| Ubiquitin | E3 ligase, Cullin, TRIM, DUB |
| Cell_Cycle | Cyclin, CDKI, SAC, APC |
| RNA_Processing | DDX, RNAP, Translation, Ribosome |
| Mitochondria | Complex I-V, Fusion, Fission, Apoptosis |
| Cytoskeleton | Actin, Tubulin, Kinesin, Dynein |
| Signaling | Wnt, Notch, Hedgehog, NFkB, RAS |